# Visualize a Cahn-Hilliard sphere run

Reads one resampled run from the Zarr dataset produced by
`datagen/cahn_hilliard/scripts/postprocess.py` and renders snapshots,
Hovmöller diagrams, conservation/separation diagnostics, and animations of
the order parameter `phi` on the sphere.

Dataset layout: `(time, field, lat, lon)` with `field = ['phi']`.
Time is in solver units (the FiPy CH equations use `D = a = 1`, so the
unit of time is `ephilon^2 / D` set by the parameter scaling).
The downstream resampling path is the unstructured kd-tree IDW route in
`datagen.resample.resample_unstructured_run`.

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`, `cartopy`, `imageio[ffmpeg]`.
All ship with the `datagen` project env. Add `ipykernel` / `jupyterlab`
if not yet installed:

```bash
uv add --project datagen ipykernel jupyterlab
uv run --project datagen jupyter lab
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# Default points at the single-trajectory production-shape test produced by
# `datagen/cahn_hilliard/slurm/single.sbatch`. Swap in any run_XXXX.zarr
# from the full sweep once it lands under
# /scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/cahn-hilliard-sphere/processed/.
DATA_PATH = "/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/cahn-hilliard-sphere/processed/run_0000.zarr"
DATA_PATH = "/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/cahn-hilliard-sphere/processed/run_0088.zarr"
#DATA_PATH = "/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/cahn-hilliard-sphere/processed/run_0576.zarr"
#DATA_PATH = "/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/cahn-hilliard-sphere/processed/run_0764.zarr"
# ^ extreme cases above

DATA_PATH = "/scicore/home/dokman0000/GROUP/PDEDatasets/SphericalPDEs/cahn-hilliard-sphere/processed/run_0192.zarr"

ds = xr.open_zarr(DATA_PATH)
ds

In [ ]:
# Quick parameter + shape summary.
print("Dims:", dict(ds.sizes))
print("Time range:", float(ds.time.min()), "..", float(ds.time.max()), "(solver units)")
print("Lat range: ", float(ds.lat.min()), "..", float(ds.lat.max()), "deg")
print("Lon range: ", float(ds.lon.min()), "..", float(ds.lon.max()), "deg")
print("Parameters:")
for k, v in ds.attrs.items():
    if k.startswith("param_"):
        print(f"  {k[6:]:14s} = {v}")

## Initial state

The initial condition is seeded Gaussian noise with mean `mean_init` and
variance `0.01`, so the first frame should look like uniform speckle
clustered around that mean (`0.35` for `run_0000`). Phase separation hasn't
started yet — the unstable spinodal mode needs a few solver-time units to
grow.


In [ ]:
ds.fields.field

In [ ]:
lat = ds.lat.values
lon = ds.lon.values
phi_all = ds.fields.sel(field="phi").values  # (time, lat, lon)

# Cahn-Hilliard's order parameter is bounded in [0, 1] with small transient
# overshoots; render with a divergent map centred on the spinodal point 0.5
# so the two phases sit at opposite ends of the colour scale.
vmin, vmax = 0.0, 1.0

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
im = ax.pcolormesh(lon, lat, phi_all[0], cmap="RdBu_r",
                   vmin=vmin, vmax=vmax, shading="auto")
ax.set_title(f"phi at t = {float(ds.time.isel(time=0)):.2f} (solver units)")
ax.set_xlabel("lon [deg]")
ax.set_ylabel("lat [deg]")
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="phi")
plt.show()

## Time evolution: spinodal decomposition and coarsening

Six panels evenly spaced through the run. Expect three regimes:

1. **Linear growth** of the dominant unstable wavelength (interfaces appear).
2. **Sharp interfaces** between phase A (`phi ≈ 0`) and phase B (`phi ≈ 1`).
3. **Coarsening**: domains grow via curvature-driven motion, so feature
   counts drop and characteristic scales increase over time.


In [ ]:
n_times = ds.sizes["time"]
panels = np.linspace(0, n_times - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True, sharex=True, sharey=True)
for ax, ti in zip(axes.flat, panels):
    im = ax.pcolormesh(lon, lat, phi_all[ti], cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=ti)):.2f}")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="phi")
fig.suptitle("Cahn-Hilliard phase evolution")
plt.show()

## Zonal-mean Hovmöller diagram

Zonal mean of `phi` over longitude as a function of `(time, lat)`. Cahn-Hilliard
on the sphere has no preferred direction, so the zonal mean stays close to
`mean_init` everywhere — any persistent banding would indicate a numerical
artefact (e.g. mesh anisotropy biasing one of the phases towards one pole).


In [ ]:
phi_zm = ds.fields.sel(field="phi").mean("lon")  # (time, lat)
t_solver = ds.time.values

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
im = ax.pcolormesh(t_solver, lat, phi_zm.T, cmap="RdBu_r",
                   vmin=vmin, vmax=vmax, shading="auto")
ax.set_title("Zonal-mean phi")
ax.set_xlabel("time (solver units)")
ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=ax, label="<phi>_lon")
plt.show()

## Globally-integrated diagnostics

Three sanity checks against the Cahn-Hilliard physics:

- **Area-weighted mean of `phi`** is conserved by the CH dynamics
  (mass conservation). Should sit at `mean_init` ± O(grid + interpolation
  error) for the whole run.
- **Area-weighted standard deviation** measures how well-separated the
  phases are. Starts near `sqrt(variance)` (the IC noise level), grows
  during spinodal decomposition, saturates near `sqrt(mean·(1-mean))` once
  interfaces are sharp.
- **Min / max envelope** shows when `phi` overshoots `[0, 1]`. Small
  transient overshoot is normal; a steady drift would suggest the
  interface width `ephilon` is too small for the mesh.


In [ ]:
lat_rad = np.deg2rad(ds.lat.values)
w_lat = np.cos(lat_rad)
w_lat = w_lat / w_lat.sum()

# Per-snapshot area-weighted moments. Average over lon (uniform weight),
# then weight rows by cos(lat).
phi_lon_mean = phi_all.mean(axis=-1)              # (time, lat)
phi_lon_mean_sq = (phi_all ** 2).mean(axis=-1)    # (time, lat)

area_mean = phi_lon_mean @ w_lat                  # (time,)
area_mean_sq = phi_lon_mean_sq @ w_lat
area_std = np.sqrt(np.clip(area_mean_sq - area_mean ** 2, 0.0, None))
phi_min = phi_all.min(axis=(-1, -2))
phi_max = phi_all.max(axis=(-1, -2))

mean_init = float(ds.attrs.get("param_mean_init", float("nan")))
sat_std = float(np.sqrt(mean_init * (1.0 - mean_init)))

fig, axes = plt.subplots(1, 3, figsize=(18, 4), constrained_layout=True)

axes[0].plot(t_solver, area_mean, label="<phi>")
axes[0].axhline(mean_init, color="k", lw=0.8, ls="--", label=f"mean_init = {mean_init:.2f}")
axes[0].set_xlabel("time (solver units)")
axes[0].set_ylabel("<phi>")
axes[0].set_title("Mass conservation")
axes[0].legend(loc="best")

axes[1].plot(t_solver, area_std, label="std(phi)")
axes[1].axhline(sat_std, color="k", lw=0.8, ls="--",
                label=f"sqrt(m(1-m)) = {sat_std:.2f}")
axes[1].set_xlabel("time (solver units)")
axes[1].set_ylabel("std(phi)")
axes[1].set_title("Phase separation")
axes[1].legend(loc="best")

axes[2].plot(t_solver, phi_min, label="min phi")
axes[2].plot(t_solver, phi_max, label="max phi")
axes[2].axhline(0.0, color="k", lw=0.8, ls=":")
axes[2].axhline(1.0, color="k", lw=0.8, ls=":")
axes[2].set_xlabel("time (solver units)")
axes[2].set_ylabel("phi")
axes[2].set_title("Min / max envelope")
axes[2].legend(loc="best")

plt.show()

## Phi animation (flat MP4)

Same canvas-grab pattern as the Mickelin notebook. `imageio[ffmpeg]`
bundles its own ffmpeg, no system install needed.


In [ ]:
import imageio.v3 as iio
from tqdm import trange

fps = 20
dpi = 100
output_path = Path("cahn_hilliard_phi.mp4")

frames = []
for i in trange(phi_all.shape[0]):
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True, dpi=dpi)
    mesh = ax.pcolormesh(lon, lat, phi_all[i], cmap="RdBu_r",
                          vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=i)):.2f} (solver units)")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    fig.colorbar(mesh, ax=ax, label="phi")

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")

## Phi animation (spherical, rotating globe)

Cartopy orthographic projection with a slow rotation around the central
longitude over the course of the run. The spherical view makes the
geodesic interface curvature obvious — droplets/strips cap into spherical
caps as they coarsen.


In [ ]:
import cartopy.crs as ccrs

fps = 20
dpi = 100
output_path = Path("cahn_hilliard_phi_spherical.mp4")

num_frames = phi_all.shape[0]
frames = []

for i in trange(num_frames):
    fig = plt.figure(figsize=(8, 8), constrained_layout=True, dpi=dpi)

    cent_lon = (i / num_frames) * 360.0 - 180.0
    ax = fig.add_subplot(
        1, 1, 1,
        projection=ccrs.Orthographic(central_longitude=cent_lon, central_latitude=20.0),
    )
    ax.set_global()
    ax.gridlines(alpha=0.3, linestyle="--")

    mesh = ax.pcolormesh(
        lon, lat, phi_all[i],
        cmap="RdBu_r",
        vmin=vmin, vmax=vmax,
        shading="auto",
        transform=ccrs.PlateCarree(),
    )
    ax.set_title(f"Cahn-Hilliard phi | t = {float(ds.time.isel(time=i)):.2f}")
    fig.colorbar(mesh, ax=ax, shrink=0.6, label="phi", pad=0.05)

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")